# Credit Card Fraud Detection: Experiment

Follows `methodology.md`. Research question: does a Random Forest beat a class-weighted
Logistic Regression on the ULB credit card dataset when judged by precision-recall metrics?

Place `creditcard.csv` (Kaggle: `mlg-ulb/creditcardfraud`) next to this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             precision_recall_curve)

SEED = 42
DATA_PATH = "creditcard.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Exploration

In [ ]:
print(df.dtypes.value_counts())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Negative amounts:", int((df["Amount"] < 0).sum()))

counts = df["Class"].value_counts()
print(counts)
print(f"Fraud rate: {df['Class'].mean():.4%}")
print(f"Accuracy of always predicting 'legitimate': {counts.max()/len(df):.4%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df.groupby("Class")["Amount"].apply(lambda s: np.log1p(s)).groupby(level=0).apply(
    lambda s: s.plot.hist(bins=40, alpha=.6, ax=ax[0], label=f"Class {s.name}"))
ax[0].set_title("log(1+Amount) by class"); ax[0].legend()
ax[1].bar(["Legit", "Fraud"], counts.sort_index().values)
ax[1].set_yscale("log"); ax[1].set_title("Class counts (log scale)")
plt.tight_layout(); plt.show()

## 2. Cleaning


In [ ]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicate rows -> {len(df)} remain")
print(f"Fraud rows remaining: {int(df['Class'].sum())}")

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["Class"], random_state=SEED)
print("Train:", train_df.shape, "fraud:", int(train_df["Class"].sum()))
print("Test: ", test_df.shape, "fraud:", int(test_df["Class"].sum()))

## 3. Feature engineering
- `Amount_log` = log1p(Amount), standardised with a scaler fitted on train only
- `Hour_sin`, `Hour_cos` from an approximate hour of day (`Time` is seconds since the first transaction)
- raw `Time` and `Amount` dropped

In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]
scaler = StandardScaler()

def build_features(frame, fit=False):
    out = frame[v_cols].copy()
    amount_log = np.log1p(frame[["Amount"]])
    out["Amount_log"] = (scaler.fit_transform(amount_log) if fit
                         else scaler.transform(amount_log)).ravel()
    hour = (frame["Time"] // 3600) % 24
    out["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["Hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return out

X_train = build_features(train_df, fit=True)
X_test = build_features(test_df)
y_train, y_test = train_df["Class"].values, test_df["Class"].values
print(X_train.shape, X_test.shape)
print("Any NaN:", bool(X_train.isna().any().any() or X_test.isna().any().any()))

## 4. Models and evaluation
Same split and metrics for every model. Primary metric: average precision (PR-AUC).

In [ ]:
results, probas = {}, {}

def evaluate(name, model):
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    probas[name] = p
    results[name] = {
        "PR-AUC (avg precision)": average_precision_score(y_test, p),
        "ROC-AUC": roc_auc_score(y_test, p),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    }
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"{name}: TP={tp} FN={fn} FP={fp} TN={tn}")
    return model

lr = evaluate("Logistic Regression",
              LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))
pd.DataFrame(results).T.round(4)

In [ ]:
rf = evaluate("Random Forest",
              RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                     min_samples_leaf=2, n_jobs=-1, random_state=SEED))
pd.DataFrame(results).T.round(4)

## 5. Comparison

In [ ]:
comparison = pd.DataFrame(results).T.round(4)
display(comparison) if "display" in globals() else print(comparison)

plt.figure(figsize=(6, 4.5))
for name, p in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, p)
    plt.plot(rec, prec, label=f"{name} (AP={results[name]['PR-AUC (avg precision)']:.3f})")
plt.axhline(y_test.mean(), ls="--", c="grey", label="No-skill baseline")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-recall curves (test set)")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Findings and what I would change

>

1. On the held-out test set, Random Forest reached PR-AUC [X] versus [Y] for Logistic Regression, so the [better model] ranks fraud higher than the other.
2. At a 0.5 threshold, Logistic Regression's recall was [R1] but its precision was only [P1], meaning many false alarms because of class weighting, while Random Forest gave recall [R2] and precision [P2].
3. Accuracy would have been above 99% for both models (and for a model that never flags fraud), which confirms it is the wrong metric here and that the papers' headline accuracies say little on their own.
4. The test set holds only about [N] frauds, so a difference of a few caught frauds moves recall a lot, and these results come from a single split and should be treated as noisy.
5. With more time I would tune the decision threshold to a target precision or cost, use stratified cross-validation or a time-based split, and try SMOTE/ADASYN applied inside the training folds only.
6. I would also add a gradient-boosting model and a 1D CNN like the reviewed papers, to test whether the deep model beats well-tuned classical ones.